<a href="https://colab.research.google.com/github/kraytenbb/ml/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F%20%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0%20%E2%84%9612%20%D1%87%D0%B0%D1%81%D1%82%D1%8C%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа. Геомаркетинговое исследование территорий с применением методов машинного обучения**


## **Цель работы**


Овладеть методами пространственного анализа и машинного обучения для решения практической задачи определения оптимальных локаций размещения коммерческих объектов.

## **Введение**


В современном бизнесе местоположение коммерческого объекта играет ключевую роль в его успешности. Геомаркетинговый анализ позволяет объективно оценить привлекательность различных локаций, опираясь на количественные показатели и алгоритмы машинного обучения.

В рамках данной работы вы примените полный цикл пространственного анализа, включая сбор данных из открытых источников, обработку и агрегацию пространственной информации, обучение моделей машинного обучения и визуализацию результатов.

## **Задание**


Провести геомаркетинговое исследование для выбора оптимальных локаций размещения новых точек определенного типа бизнеса на территории выбранного города.

## **Порядок выполнения работы**

### **Часть 1. Подготовка данных и определение задачи**



1. **Индивидуальный выбор территории и типа бизнеса**:
   - Выберите город/район для проведения анализа (например, центральная часть Москвы, Санкт-Петербурга или любого другого крупного города)
   - Определите тип бизнеса для анализа (аптеки, продуктовые магазины, пункты выдачи заказов, рестораны определенной кухни и т.д.)
   - Обоснуйте свой выбор: почему данная территория и тип бизнеса интересны для анализа?

2. **Определение ключевых факторов успешности**:
   - Самостоятельно сформулируйте не менее 5 факторов, которые могут влиять на успешность выбранного типа бизнеса
   - Для каждого фактора определите, какими данными из OpenStreetMap его можно количественно описать
   - Составьте таблицу соответствия между факторами и тегами OpenStreetMap

3. **Сбор исходных данных**:
   - Настройте необходимые библиотеки из теоретического материала
   - Определите необходимую область интереса (ROI) с помощью интерактивной карты
   - Загрузите данные о существующих объектах вашего типа бизнеса и объектах инфраструктуры, связанных с выделенными вами факторами

In [3]:
!pip -q install osmnx h3 folium scikit-learn

import osmnx as ox
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import folium

ox.settings.use_cache = True
ox.settings.log_console = False

place = "Арбат, Москва, Россия"

business = "pharmacy"

tags = {
    "amenity": [
        "pharmacy",
        "restaurant",
        "cafe",
        "bank",
        "school",
        "hospital",
        "bus_station",
        "fast_food"
    ],
    "shop": [
        "supermarket",
        "convenience"
    ],
    "leisure": [
        "park"
    ]
}


area = ox.geocode_to_gdf(place)

polygon = area.geometry.iloc[0]


objects = ox.features_from_polygon(polygon, tags)

objects = objects.reset_index()
objects = objects[objects.geometry.notna()].copy()

projected_crs = area.estimate_utm_crs()

objects_projected = objects.to_crs(projected_crs)

objects_projected["geometry"] = objects_projected.geometry.centroid

objects = objects_projected.to_crs("EPSG:4326")



points = objects[
    ["geometry", "amenity", "shop", "leisure"]
].copy()

points["amenity"] = points["amenity"].fillna("")
points["shop"] = points["shop"].fillna("")
points["leisure"] = points["leisure"].fillna("")

area_projected = area.to_crs(projected_crs)

center = area_projected.geometry.iloc[0].centroid

center_gdf = gpd.GeoSeries(
    [center],
    crs=projected_crs
).to_crs("EPSG:4326")

center = center_gdf.iloc[0]

esri_tiles = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
)

roi_map = folium.Map(
    location=[center.y, center.x],
    zoom_start=14,
    tiles=esri_tiles,
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors"
)

folium.GeoJson(
    area[["geometry"]],
    name="Область"
).add_to(roi_map)


roi_map

### **Часть 2. Пространственное агрегирование и создание признаков**


1. **Создание гексагональной сетки**:
   - Самостоятельно определите оптимальную детализацию сетки H3 для вашего анализа
   - Обоснуйте выбор разрешения (resolution) сетки с учётом масштаба вашей территории и специфики бизнеса
   - Покройте территорию гексагональной сеткой выбранного разрешения

2. **Инженерия пространственных признаков**:
   - Определите радиусы для анализа ближнего и среднего окружения (могут отличаться от предложенных в теории в зависимости от специфики вашего бизнеса)
   - Разработайте и реализуйте не менее 10 пространственных признаков, описывающих характеристики каждой ячейки
   - **Дополнительное задание**: придумайте и реализуйте не менее 2 пространственных признаков, которых нет в теоретическом материале

3. **Анализ полученных признаков**:
   - Рассчитайте базовую статистику по каждому признаку
   - Исследуйте корреляции между признаками
   - Выявите и обработайте выбросы и пропущенные значения, если они имеются

In [4]:
from shapely.geometry import Polygon


resolution = 9
cells = h3.geo_to_cells(polygon.__geo_interface__, resolution)

grid = gpd.GeoDataFrame(
    {"h3": cells},
    geometry=[
        Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(cell)])
        for cell in cells
    ],
    crs="EPSG:4326"
)

categories = {
    "pharmacy_count": ("amenity", "pharmacy"),
    "restaurant_count": ("amenity", "restaurant"),
    "cafe_count": ("amenity", "cafe"),
    "bank_count": ("amenity", "bank"),
    "school_count": ("amenity", "school"),
    "hospital_count": ("amenity", "hospital"),
    "bus_count": ("amenity", "bus_station"),
    "fast_food_count": ("amenity", "fast_food"),
    "supermarket_count": ("shop", "supermarket"),
    "convenience_count": ("shop", "convenience"),
    "park_count": ("leisure", "park")
}

points = objects.copy()

for name, (column, value) in categories.items():
    mask = points[column].eq(value)
    coords = points.loc[mask, "geometry"]
    counts = []
    for cell in grid["h3"]:
        lat, lon = h3.cell_to_latlng(cell)
        center = gpd.GeoSeries(
            [gpd.points_from_xy([lon], [lat])[0]],
            crs="EPSG:4326"
        )
        if len(coords) == 0:
            counts.append(0)
        else:
            distances = coords.to_crs("EPSG:3857").distance(
                center.to_crs("EPSG:3857").iloc[0]
            )
            counts.append(int((distances <= 1000).sum()))
    grid[name] = counts

grid["amenity_diversity"] = (
    (grid["restaurant_count"] > 0).astype(int)
    + (grid["cafe_count"] > 0).astype(int)
    + (grid["bank_count"] > 0).astype(int)
    + (grid["school_count"] > 0).astype(int)
    + (grid["hospital_count"] > 0).astype(int)
    + (grid["bus_count"] > 0).astype(int)
    + (grid["fast_food_count"] > 0).astype(int)
    + (grid["supermarket_count"] > 0).astype(int)
    + (grid["convenience_count"] > 0).astype(int)
    + (grid["park_count"] > 0).astype(int)
)

grid["accessibility_index"] = (
    grid["bus_count"] * 2
    + grid["supermarket_count"]
    + grid["restaurant_count"]
    + grid["cafe_count"]
    + grid["bank_count"]
)

feature_columns = [
    "restaurant_count", "cafe_count", "bank_count", "school_count",
    "hospital_count", "bus_count", "fast_food_count", "supermarket_count",
    "convenience_count", "park_count", "amenity_diversity",
    "accessibility_index"
]

display(grid[feature_columns].describe())


,restaurant_count,cafe_count,bank_count,school_count,hospital_count,bus_count,fast_food_count,supermarket_count,convenience_count,park_count,amenity_diversity,accessibility_index
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.0,21.000000,21.000000,21.000000,21.000000,21.000000,21.00000
mean,35.142857,23.714286,9.476190,2.142857,0.523810,0.0,12.142857,3.142857,5.142857,4.619048,8.142857,71.47619
std,15.408717,12.798995,3.385543,1.590148,0.511766,0.0,7.226934,1.740279,2.329929,2.597618,1.014185,30.21195
min,13.000000,4.000000,2.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,0.000000,5.000000,21.00000
25%,25.000000,13.000000,7.000000,1.000000,0.000000,0.0,7.000000,2.000000,3.000000,2.000000,8.000000,52.00000
50%,33.000000,21.000000,11.000000,2.000000,1.000000,0.0,11.000000,4.000000,5.000000,5.000000,8.000000,74.00000
75%,48.000000,36.000000,12.000000,3.000000,1.000000,0.0,17.000000,4.000000,7.000000,6.000000,9.000000,91.00000
max,63.000000,48.000000,14.000000,5.000000,1.000000,0.0,28.000000,7.000000,9.000000,9.000000,9.000000,126.00000


### **Часть 3. Моделирование привлекательности локаций**



1. **Подготовка целевой переменной**:
   - Определите, как будет сформирована целевая переменная для вашей задачи (по умолчанию: наличие объектов выбранного типа в ячейке)
   - Исследуйте распределение целевой переменной и оцените её сбалансированность
   - При необходимости, предложите стратегию работы с несбалансированными данными

2. **Разработка моделей машинного обучения**:
   - Реализуйте и обучите несколько моделей (минимум 2) для предсказания привлекательности локации
   - Проведите оценку важности признаков для каждой модели
   - Сравните модели по метрикам качества и выберите наилучшую
   - **Дополнительное задание**: настройте гиперпараметры модели с помощью поиска по сетке (GridSearchCV) или случайного поиска (RandomSearchCV)

3. **Улучшение модели с помощью кластеризации**:
   - Выполните кластеризацию ячеек по их характеристикам
   - Определите оптимальное число кластеров с помощью метода локтя или силуэта
   - Визуализируйте результаты кластеризации
   - Проверьте, улучшает ли добавление информации о кластерах качество основной модели

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

data = grid[feature_columns + ["pharmacy_count"]].copy()
data["target"] = (data["pharmacy_count"] > 0).astype(int)

X = data[feature_columns]
y = data["target"]

print(y.value_counts())
print(y.value_counts(normalize=True))

models = {}
results = []
importance = {}

# Классификация возможна только при наличии минимум двух классов
if y.nunique() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    models = {
        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000)
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    }

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        results.append([
            name,
            accuracy_score(y_test, pred),
            precision_score(y_test, pred, zero_division=0),
            recall_score(y_test, pred, zero_division=0),
            f1_score(y_test, pred, zero_division=0)
        ])

        if name == "RandomForest":
            importance[name] = pd.Series(
                model.feature_importances_,
                index=feature_columns
            ).sort_values(ascending=False)
        else:
            importance[name] = pd.Series(
                np.abs(model.named_steps["logisticregression"].coef_[0]),
                index=feature_columns
            ).sort_values(ascending=False)

    results_df = pd.DataFrame(
        results,
        columns=["model", "accuracy", "precision", "recall", "f1"]
    )

    display(results_df)
    display(importance["RandomForest"].head(10))

else:
    print("В целевой переменной только один класс.")
    print("Классификация пропущена, так как обучить модель невозможно.")

# Кластеризация выполняется независимо от классификации
cluster_data = StandardScaler().fit_transform(X)
scores = {}

max_k = min(7, len(data))

for k in range(2, max_k):
    labels = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    ).fit_predict(cluster_data)

    scores[k] = silhouette_score(cluster_data, labels)

if scores:
    best_k = max(scores, key=scores.get)

    kmeans = KMeans(
        n_clusters=best_k,
        random_state=42,
        n_init=10
    )

    data["cluster"] = kmeans.fit_predict(cluster_data)

    print("Лучшее число кластеров:", best_k)
    print("Лучшее значение silhouette:", scores[best_k])

    display(
        data.groupby("cluster")[feature_columns]
        .mean()
        .round(2)
    )

target
1    21
Name: count, dtype: int64
target
1    1.0
Name: proportion, dtype: float64
В целевой переменной только один класс.
Классификация пропущена, так как обучить модель невозможно.
Лучшее число кластеров: 2
Лучшее значение silhouette: 0.3877756927957363


,restaurant_count,cafe_count,bank_count,school_count,hospital_count,bus_count,fast_food_count,supermarket_count,convenience_count,park_count,amenity_diversity,accessibility_index
cluster,,,,,,,,,,,,
0,43.69,31.54,11.77,3.00,0.69,0.0,16.46,3.92,6.15,6.00,8.69,90.92
1,21.25,11.00,5.75,0.75,0.25,0.0,5.12,1.88,3.50,2.38,7.25,39.88


### **Часть 4. Расчет потенциала локаций и финальные рекомендации**



1. **Разработка интегрального показателя потенциала**:
   - Самостоятельно определите веса для факторов привлекательности среды и конкуренции
   - Обоснуйте выбранные веса в контексте вашего бизнеса
   - Рассчитайте итоговый потенциал для всех ячеек сетки
   - Категоризируйте потенциал для упрощения интерпретации результатов

2. **Визуализация результатов**:
   - Создайте интерактивную карту с тепловым слоем потенциала
   - Добавьте маркеры существующих объектов вашего типа бизнеса
   - Выделите топ-10 локаций с наивысшим потенциалом
   - Подготовьте отдельную карту фокуса на лучших локациях

3. **Формирование бизнес-рекомендаций**:
   - Составьте список из 5-7 конкретных локаций для размещения новых объектов
   - Для каждой рекомендуемой локации укажите:
     * Точные координаты
     * Значение потенциала
     * Ключевые характеристики локации
     * Преимущества и возможные риски размещения в данной точке
   - Подготовьте общие рекомендации по стратегии территориального развития для выбранного бизнеса

In [11]:
grid["environment_score"] = (
    grid["amenity_diversity"] * 0.35
    + grid["accessibility_index"] * 0.45
    + grid["supermarket_count"] * 0.20
)

grid["competition_score"] = grid["pharmacy_count"] / (
    grid["pharmacy_count"].max() + 1
)

grid["potential"] = (
    grid["environment_score"] * 0.8
    - grid["competition_score"] * 0.2
)

grid["potential_category"] = pd.qcut(
    grid["potential"].rank(method="first"),
    q=4,
    labels=["Низкий", "Средний", "Высокий", "Очень высокий"]
)

top10 = grid.nlargest(10, "potential").copy()

print("Топ-10 локаций:")

display(
    top10[
        ["h3", "potential", "potential_category", "cluster"] + feature_columns
    ].round(3)
)

esri_tiles = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
)

result_map = folium.Map(
    location=[center.y, center.x],
    zoom_start=14,
    tiles=esri_tiles,
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors"
)

folium.GeoJson(
    area[["geometry"]],
    name="Область"
).add_to(result_map)

for _, row in grid.iterrows():
    lat, lon = h3.cell_to_latlng(row["h3"])
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        popup=f"Потенциал: {row['potential']:.2f}<br>Кластер: {row['cluster']}",
        fill=True
    ).add_to(result_map)

for _, row in top10.iterrows():
    lat, lon = h3.cell_to_latlng(row["h3"])
    folium.Marker(
        location=[lat, lon],
        popup=f"Рекомендованная локация<br>Потенциал: {row['potential']:.2f}"
    ).add_to(result_map)

result_map

Топ-10 локаций:


,h3,potential,potential_category,cluster,restaurant_count,cafe_count,bank_count,school_count,hospital_count,bus_count,fast_food_count,supermarket_count,convenience_count,park_count,amenity_diversity,accessibility_index
7,8911aa7ae2fffff,48.491,Очень высокий,0,61,48,12,4,1,0,28,5,8,6,9,126
12,8911aa7ae23ffff,46.291,Очень высокий,0,63,37,12,3,1,0,21,7,9,6,9,119
20,8911aa7ae27ffff,40.953,Очень высокий,0,57,32,11,5,1,0,21,5,7,9,9,105
9,8911aa7a85bffff,40.827,Очень высокий,0,48,39,14,5,1,0,18,4,6,6,9,105
1,8911aa7aa97ffff,37.598,Очень высокий,0,39,40,13,4,1,0,17,4,5,7,9,96
10,8911aa7a84bffff,35.787,Высокий,0,33,40,14,2,1,0,21,4,4,7,9,91
5,8911aa7ae2bffff,35.136,Высокий,0,48,28,10,2,0,0,16,4,7,2,8,90
11,8911aa7ae33ffff,34.416,Высокий,0,51,22,11,2,0,0,10,4,8,5,8,88
4,8911aa7ae3bffff,31.558,Высокий,0,45,20,11,2,0,0,11,4,7,3,8,80
18,8911aa7ae37ffff,30.118,Высокий,0,39,21,12,4,0,0,11,4,8,9,8,76


## **Рекомендации по выполнению**


1. Начните с малой территории для тестирования кода и методологии, затем расширяйте анализ.
2. Используйте инкрементальный подход: сначала реализуйте базовый функционал, затем улучшайте его.
3. Регулярно сохраняйте промежуточные результаты работы.
4. При выборе признаков опирайтесь не только на учебный материал, но и на научные статьи по геоанализу и геомаркетингу.
5. Обращайте внимание на особенности территории и уникальные характеристики выбранного бизнеса.
6. Для оценки результатов старайтесь сопоставить их с реальным расположением успешных объектов аналогичного бизнеса.